# Compare three enemy-missile forecasting architectures

Run notebook **17**, then train the three families with **18** (ordered dense
projection), **19** (unordered friendly/enemy missile-set encoder, no messages)
and **20** (typed aircraft–missile graph encoder). This notebook reads their saved
predictions and diagnostics; it neither trains a model nor loads the large
prepared feature array. Run All writes a fresh timestamped comparison directory.

The default inputs are the completed-family receipts saved by those notebooks.
For a reproducible historical comparison, enter explicit run paths below or save
the receipts used. All paths are recorded in the comparison manifest. The helper
requires all three families, the same dataset digest, class/feature ordering,
forecast horizons, matched seed sets and protocol, plus identical sample indices
and targets. It rejects incompatible or incomplete inputs rather than silently
comparing different populations. Architectural encoder settings and parameter
counts can differ and remain reported.

"Test" is the checkpoint-selection set; "validation" is held-out descriptive
reporting. Tables are not used to select a validation winner, seed or ensemble.
If validation has guided development, subsequent generalisation claims need a new
independent evaluation set. Bootstrap intervals resample complete scenario groups
when the saved diagnostic sample metadata permits it, not overlapping windows.
These are conditional intervals for the saved models, without search correction.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/enemy_missile_diagnostics.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from bvr_behavior_prediction.paired_skill_forecasting import read_json, write_json, sha256
from bvr_behavior_prediction.enemy_missile_diagnostics import compare_architectures

EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_enemy_missile_forecasting"
FAMILIES = ("transformer", "missile_set", "typed_graph")
# Fill all three lists to pin historical runs. Empty mapping reads completed receipts.
EXPLICIT_RUNS = {}  # e.g. {"transformer": [r"C:\...\run_s42", ...], ...}
BOOTSTRAP_RESAMPLES = 1000  # Whole-scenario resampling; 0 disables intervals.
RUNS = {}
RECEIPTS = {}
if EXPLICIT_RUNS:
    if set(EXPLICIT_RUNS) != set(FAMILIES):
        raise ValueError("EXPLICIT_RUNS must name transformer, missile_set and typed_graph.")
    RUNS = {family: [Path(p).expanduser().resolve() for p in paths]
            for family, paths in EXPLICIT_RUNS.items()}
else:
    for family in FAMILIES:
        receipt_path = EXPERIMENT_ROOT / "comparisons" / f"latest_{family}_runs.json"
        if not receipt_path.is_file():
            raise FileNotFoundError(f"Missing {receipt_path}. Finish notebook 18/19/20, "
                                    "or supply EXPLICIT_RUNS for all three families.")
        receipt = read_json(receipt_path)
        if receipt["architecture"] != family:
            raise ValueError(f"Wrong architecture in {receipt_path}")
        RECEIPTS[family] = receipt
        RUNS[family] = [Path(p).expanduser().resolve() for p in receipt["runs"]]
if not all(RUNS.values()):
    raise ValueError("Each architecture needs at least one completed seed run.")
STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = EXPERIMENT_ROOT / "comparisons" / f"three_architectures_{STAMP}"
print("Saved comparison destination:", OUTPUT)
display(pd.DataFrame([{"architecture": family, "run": str(run)}
                      for family, paths in RUNS.items() for run in paths]))


## Verify and compare the frozen prediction artifacts

The tables retain every configured seed. Horizon-level and equal-horizon means
include NLL, accuracy, class F1, calibration and parameter counts. Per-class deltas
show which catalogue entries change and their support. Paired scenario intervals
compare each challenger to the ordered Transformer, and the graph directly to
the missile-set model, on the same examples. `horizon_means` contains each seed's
equal-horizon mean; `horizon_mean_aggregate` summarises those means across seeds.
Seed variability and scenario intervals describe different uncertainty sources.
Comparison outputs are written to CSV/PNG/JSON, independent of notebook output.


In [ ]:
%%time
TABLES = compare_architectures(RUNS, OUTPUT, bootstrap_resamples=BOOTSTRAP_RESAMPLES)
write_json(OUTPUT / "input_run_lists.json", {
    "runs": {family: [str(p) for p in paths] for family, paths in RUNS.items()},
    "receipts": RECEIPTS, "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
})
# Include the notebook's input receipt in the complete comparison artifact audit.
comparison_manifest = read_json(OUTPUT / "manifest.json")
comparison_manifest["artifacts"]["input_run_lists.json"] = sha256(OUTPUT / "input_run_lists.json")
write_json(OUTPUT / "manifest.json", comparison_manifest)
for name, table in TABLES.items():
    if isinstance(table, pd.DataFrame):
        print(name)
        display(table.round(4))
print("Saved comparison manifest:", OUTPUT / "manifest.json")


In [ ]:
%%time
# Display the plots that were actually persisted by the comparison helper.
for path in sorted(OUTPUT.glob("*.png")):
    display(Image(filename=str(path)))
print("All comparison artifacts:", OUTPUT)


## Interpretation

Compare the set encoder with the dense baseline to assess ordering/pooling changes;
compare the graph with the set encoder to assess whether explicit relational
messages add value. Their frame encoders differ in capacity: parameter counts and
training duration are reported, not assumed matched. Class weighting and L1 are
held fixed; summed L1 can still differ with parameter count. NLL and rare-class F1
may move in different directions. Retain unsuccessful seeds and unsupported-class
audits. These experiments forecast recorded catalogue selections; identical executed
controls can have distinct names, so label gains are not automatically physical
maneuver gains. No measured improvement is claimed before training these models.
